# ValueLens - data volume check

Run this in the **same Fabric workspace as your Lakehouse**, with the Lakehouse
attached as the notebook's default (Explorer -> Add -> existing Lakehouse).

It reads only. It writes nothing and changes nothing.

It answers three questions:

1. how many licensed users the dashboard will actually count, and why any are excluded
2. what date range your audit data really covers
3. whether the Agents 365 table has landed and which columns are populated

Run all cells and review the output locally before sharing any tenant data.

Licence flags describe the imported assigned-product snapshot, not enabled Copilot service plans. For Microsoft 365 E7, update the Licensed Users Direct Ingester (including intentional SKU-pattern overrides), rerun it, then the Audit Log Processor, then refresh Power BI. This diagnostic does not reclassify products or change flags.


In [ ]:
# CONFIG - only change these if your tables live elsewhere
LICENSED_TABLE = 'copilot_licensed_users'
AUDIT_TABLE    = 'copilot_interactions_curated'
AGENTS_TABLE   = 'agents_365'
SCHEMA         = None      # e.g. 'dbo' for a schema-enabled Lakehouse; None to auto-detect


In [ ]:
# === resolve table names ======================================================
from pyspark.sql import functions as F

def _resolve(name):
    """Find <name> whether the Lakehouse is schema-enabled or not."""
    cands = []
    if SCHEMA:
        cands.append(f'{SCHEMA}.{name}')
    cands += [name, f'dbo.{name}']
    for c in cands:
        try:
            spark.read.table(c).limit(1).count()
            return c
        except Exception:
            continue
    return None

resolved = {k: _resolve(v) for k, v in
            {'licensed': LICENSED_TABLE, 'audit': AUDIT_TABLE, 'agents': AGENTS_TABLE}.items()}
for k, v in resolved.items():
    print(f'{k:10} -> {v or "NOT FOUND"}')


def _pick(df, exact, contains=None, exclude=()):
    """
    Resolve a column by exact name first, then by substring.

    Exact-first matters: a fuzzy 'licen' match hits
    'Exchange_License_Assign_Date' long before 'Has_license'.
    """
    lower = {c.lower(): c for c in df.columns}
    for name in exact:
        if name.lower() in lower:
            return lower[name.lower()]
    if contains:
        for c in df.columns:
            cl = c.lower()
            if any(x in cl for x in contains) and not any(x in cl for x in exclude):
                return c
    return None


LICENCE_NAMES = ['Has license', 'Has_license', 'HasLicense', 'HasCopilot',
                 'Has Copilot', 'Has_Copilot', 'Has Copilot License']
LICENCE_EXCLUDE = ['assign', 'activity', 'exchange', 'onedrive', 'sharepoint',
                   'skype', 'yammer', 'teams', 'date']

UPN_NAMES = ['UPN_Normalized', 'User Principal Name', 'User_Principal_Name',
             'userPrincipalName', 'UserPrincipalName']

AUDIT_USER_NAMES = ['Audit_UserId', 'UserId', 'UserPrincipalName',
                    'User Principal Name', 'UPN_Normalized']

DATE_NAMES = ['CreationDate', 'ActivityDate', 'CreatedDateTime']


In [ ]:
# === 1. LICENSED USERS  (drives "Total Licensed Users") =======================
# This measure is NOT affected by the date slicer or by incremental refresh.
# A low number here means the table itself is short, or the "Has license" values
# are not ones the dashboard recognises.
from collections import Counter

ACCEPTED = {'YES', 'TRUE', 'Y', '1'}


def _accepted_flag(value):
    return value is not None and str(value).strip().upper() in ACCEPTED


def _summarize_flag_values(values, limit=25):
    counts = Counter(values)
    ordered = counts.most_common()
    counted = sum(count for value, count in ordered if _accepted_flag(value))
    return {
        'total': sum(counts.values()),
        'counted': counted,
        'display_rows': ordered[:limit],
    }


def _licensed_upns(rows, upn_col, lic_col):
    if not upn_col or not lic_col:
        return None
    out = set()
    for row in rows:
        if not _accepted_flag(row.get(lic_col)):
            continue
        norm = (row.get(upn_col) or '').strip().lower()
        if norm:
            out.add(norm)
    return out


t = resolved['licensed']
if not t:
    print(f'{LICENSED_TABLE} not found - the licensed users file has not been landed.')
else:
    df = spark.read.table(t)
    row_count = df.count()
    print(f'table   : {t}')
    print(f'rows    : {row_count:,}')
    print(f'columns : {df.columns}')
    print('Licence flags are snapshot product matches; disabled/provisioning service plans are not verified.')

    products = _pick(df, ['Assigned Products', 'Assigned_Products'])
    lic = _pick(df, LICENCE_NAMES, ['licen', 'copilot'], LICENCE_EXCLUDE)
    upn = _pick(df, UPN_NAMES, ['principal', 'upn'])

    if products and lic:
        print('\nAssigned-product combinations and stored flags (first 25 rows shown; totals are computed from the full table):')
        df.groupBy(products, lic).count().orderBy(F.desc('count')).show(25, truncate=False)

    if not lic:
        print('\n** no "Has license" column found - licensed-user totals are unknown until the flag is landed')
    else:
        flag_counts = [(row[lic], row['count']) for row in df.groupBy(lic).count().orderBy(F.desc('count')).collect()]
        summary = _summarize_flag_values([value for value, count in flag_counts for _ in range(count)])
        print(f'\ndistinct values in "{lic}" (first {len(summary["display_rows"])} shown):')
        for value, count in summary['display_rows']:
            print(f'    {str(value)!r:24} {count:>8}   {"counted" if _accepted_flag(value) else "NOT counted"}')
        print(f'\ncounted as licensed : {summary["counted"]:,} of {summary["total"]:,}')
        if summary['counted'] < summary['total']:
            print(f'** {summary["total"] - summary["counted"]:,} rows excluded because the value is not one of')
            print('   YES / TRUE / Y / 1 (after upper-casing and trimming)')

    if upn:
        blank = df.filter(F.col(upn).isNull() | (F.trim(F.col(upn)) == '')).count()
        distinct = (df.filter(F.col(upn).isNotNull() & (F.trim(F.col(upn)) != ''))
                      .select(F.lower(F.trim(F.col(upn)))).distinct().count())
        print(f'\nblank UPN rows dropped     : {blank:,}')
        print(f'distinct UPNs after dedupe : {distinct:,}')
        if lic:
            qual = (df.filter(F.upper(F.trim(F.col(f'`{lic}`'))).isin(list(ACCEPTED)))
                      .filter(F.col(f'`{upn}`').isNotNull() & (F.trim(F.col(f'`{upn}`')) != ''))
                      .select(F.lower(F.trim(F.col(f'`{upn}`')))).distinct().count())
            print(f'qualifying distinct UPNs   : {qual:,}')
            print(f'** the dashboard should show about {qual:,} total licensed users')
            if qual != distinct:
                print(f'   ({distinct - qual:,} users are in the file but not counted as licensed)')
        else:
            print('** distinct UPNs are shown above, but the dashboard total is unknown until the licence flag exists')


In [ ]:
# === 2. AUDIT LOG COVERAGE  (drives "Active Licensed Users") ==================
# This IS filtered by the RangeStart / RangeEnd parameters and by the report's
# date slicer. If the slicer sits outside the range below, activity reads as zero.
t = resolved['audit']
if not t:
    print(f'{AUDIT_TABLE} not found - the audit processor has not run.')
else:
    df = spark.read.table(t)
    n = df.count()
    print(f'table : {t}')
    print(f'rows  : {n:,}')

    dcol = _pick(df, DATE_NAMES, ['creationdate', 'activitydate'])
    if n == 0:
        print('\n** NO AUDIT ROWS - date coverage cannot be assessed.')
    elif dcol:
        agg = df.select(F.min(dcol).alias('lo'), F.max(dcol).alias('hi')).collect()[0]
        days = df.select(F.to_date(F.col(dcol))).distinct().count()
        print(f'\n"{dcol}" range : {agg["lo"]}  ->  {agg["hi"]}')
        print(f'distinct days  : {days:,}')
        print('\n** set the dashboard date slicer inside this range, or it will show zero.')
        print('   rows per month:')
        (df.groupBy(F.date_format(F.col(dcol), 'yyyy-MM').alias('month'))
           .count().orderBy('month').show(36, False))

    ucol = _pick(df, AUDIT_USER_NAMES, ['userid', 'principal'])
    if ucol:
        du = (df.filter(F.col(f'`{ucol}`').isNotNull() & (F.trim(F.col(f'`{ucol}`')) != ''))
                .select(F.lower(F.trim(F.col(f'`{ucol}`')))).distinct().count())
        print(f'distinct usable users in audit data : {du:,}')
        print('** "Active Licensed Users" can never exceed the overlap between this')
        print('   set and the licensed-users list above.')


In [ ]:
# === 3. AGENTS 365 ============================================================
t = resolved['agents']
if not t:
    print(f'{AGENTS_TABLE} not found - the Agent 365 lander/ingester has not run.')
else:
    df = spark.read.table(t)
    n = df.count()
    print(f'table   : {t}')
    print(f'rows    : {n:,}')
    print(f'columns : {len(df.columns)}')
    if n:
        print('\npopulated columns:')
        exprs = [F.count(F.when(F.col(f'`{c}`').isNotNull() &
                                (F.trim(F.col(f'`{c}`').cast('string')) != ''), 1)).alias(c)
                 for c in df.columns]
        row = df.agg(*exprs).collect()[0].asDict()
        blank = []
        for c in df.columns:
            v = row[c]
            if v:
                print(f'    {c:44} {v:>7,}/{n:,}')
            else:
                blank.append(c)
        if blank:
            print(f'\nempty ({len(blank)}):')
            for c in blank:
                print(f'    {c}')
            print('\n** empty columns may be unavailable on the chosen Agent 365 source')
            print('   (registry vs observability) or may reflect a renamed export field.')


In [ ]:
# === 4. OVERLAP CHECK =========================================================
# "Active Licensed Users" counts users present in BOTH tables. A mismatch in
# formatting (domain, casing, guest UPNs) shows up here as a small overlap.
def _overlap_diagnostic(licensed_users, audit_users, overlap, audit_rows):
    missing = []
    if audit_rows == 0:
        missing.append('NO AUDIT ROWS - check source activity, the selected ingestion window, and the audit ingester/processor outputs.')
    elif audit_users == 0:
        missing.append('NO USABLE AUDIT USER IDENTIFIERS - audit rows exist, but user identifiers are null or blank.')
    if licensed_users == 0:
        missing.append('NO LICENSED USERS WITH USABLE UPNs - check the licensed-user snapshot, licence flags, and blank UPNs.')
    if missing:
        return 'not_comparable', '\n** ' + '\n** '.join(missing) + (
            '\n   "Active Licensed Users" will be zero. Identity overlap cannot be assessed until both sources have comparable users.'
        )
    if overlap == 0:
        return 'no_overlap', (
            '\n** NO OVERLAP - "Active Licensed Users" will be zero.'
            '\n   Both sources contain usable users, but none match. Check whether the audit users are licensed,'
            '\n   then compare identity formats and tenant/environment selection; these are possible causes, not a diagnosis.'
            '\n   Compare the samples below - they must match to join.'
        )
    if overlap < min(licensed_users, audit_users) * 0.5:
        return 'low_overlap', '\n** low overlap - the two sources may format identity differently.'
    return 'overlap', ''


lt, at = resolved['licensed'], resolved['audit']
if lt and at:
    ldf, adf = spark.read.table(lt), spark.read.table(at)
    upn = _pick(ldf, UPN_NAMES, ['principal', 'upn'])
    lic = _pick(ldf, LICENCE_NAMES, ['licen', 'copilot'], LICENCE_EXCLUDE)
    ucol = _pick(adf, AUDIT_USER_NAMES, ['userid', 'principal'])
    if upn and lic and ucol:
        L = (ldf.filter(F.upper(F.trim(F.col(f'`{lic}`'))).isin(list(ACCEPTED)))
                .filter(F.col(f'`{upn}`').isNotNull() & (F.trim(F.col(f'`{upn}`')) != ''))
                .select(F.lower(F.trim(F.col(f'`{upn}`'))).alias('u')).distinct())
        A = (adf.filter(F.col(f'`{ucol}`').isNotNull() & (F.trim(F.col(f'`{ucol}`')) != ''))
                .select(F.lower(F.trim(F.col(f'`{ucol}`'))).alias('u')).distinct())
        nl, na = L.count(), A.count()
        both = L.join(A, 'u', 'inner').count()
        print(f'licensed UPNs      : {nl:,}')
        print(f'audit users        : {na:,}')
        print(f'present in both    : {both:,}')
        status, message = _overlap_diagnostic(nl, na, both, adf.count())
        if message:
            print(message)
        if status in ('no_overlap', 'low_overlap'):
            print('   sample licensed:')
            for r in L.limit(3).collect():
                print(f'      {r["u"]}')
            print('   sample audit:')
            for r in A.limit(3).collect():
                print(f'      {r["u"]}')
    elif upn and not lic:
        print('licensed flag column missing - overlap with "Active Licensed Users" is unknown.')
    else:
        print('could not find a UPN column in one of the tables')
else:
    print('need both tables for the overlap check')
